In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import math
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## EXAMPLES OF BYTE COMPARISONS

In [7]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [8]:
# no letters in common
w1b & w2b

0

In [9]:
# letters in common
w1b & w3b

147456

In [10]:
# bitwise or
w1b | w2b

673975

In [11]:
# this is the same as directly above
byte_encode_words('abhorcleft')

673975

In [12]:
byte_encode_words(ascii_lowercase)

67108863

# BUILD LEVEL 2

In [13]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

(3213696, 3)


In [14]:
l2_df['l2'].unique().shape

(640023,)

In [15]:
# BUILD THE CHAR MATRIX AND THE SELECTOR

In [16]:
l2_df.head()

,w1b,w2b,l2
0,20491,264468,284959
1,20491,532756,553247
2,20491,788500,808991
3,20491,794644,815135
4,20491,1114388,1134879


In [17]:
l2_df['w1'] = l2_df['w1b'].map(word_byte_to_word_dict)
l2_df['w2'] = l2_df['w2b'].map(word_byte_to_word_dict)

In [18]:
l2_df['l2_words'] = l2_df['w1'] + l2_df['w2']

In [19]:
l2_df.head()

,w1b,w2b,l2,w1,w2,l2_words
0,20491,264468,284959,abdom,ceils,abdomceils
1,20491,532756,553247,abdom,ceint,abdomceint
2,20491,788500,808991,abdom,celts,abdomcelts
3,20491,794644,815135,abdom,cents,abdomcents
4,20491,1114388,1134879,abdom,cequi,abdomcequi


In [20]:
letter_dict = {l:p for p,l in enumerate(ascii_lowercase)}

In [21]:
char_matrix = np.zeros(shape = (l2_df.shape[0], 26), dtype = np.int8)
for i_l2, l2 in enumerate(l2_df['l2_words']):    
    for l in l2:
        char_matrix[i_l2, letter_dict[l]] += 1

    if i_l2 % 10000 == 0:
        print(i_l2)

0
10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000
230000
240000
250000
260000
270000
280000
290000
300000
310000
320000
330000
340000
350000
360000
370000
380000
390000
400000
410000
420000
430000
440000
450000
460000
470000
480000
490000
500000
510000
520000
530000
540000
550000
560000
570000
580000
590000
600000
610000
620000
630000
640000
650000
660000
670000
680000
690000
700000
710000
720000
730000
740000
750000
760000
770000
780000
790000
800000
810000
820000
830000
840000
850000
860000
870000
880000
890000
900000
910000
920000
930000
940000
950000
960000
970000
980000
990000
1000000
1010000
1020000
1030000
1040000
1050000
1060000
1070000
1080000
1090000
1100000
1110000
1120000
1130000
1140000
1150000
1160000
1170000
1180000
1190000
1200000
1210000
1220000
1230000
1240000
1250000
1260000
1270000
1280000
1290000
1300000
1310000
1320000
1330000
1340000
1350000
1360000
1370000
1380000
13

In [22]:
# what are the indices of words that do no have the letter b?
for l, p in letter_dict.items():
    idx = char_matrix[:, p] == 0
    curr_l2_list = l2_list[idx, :]
    unique_curr_l2_list = np.unique(curr_l2_list[:, 2])
    print(l, curr_l2_list.shape, unique_curr_l2_list.shape)


a (871979, 3) (227913,)
b (2278449, 3) (437166,)
c (1899697, 3) (393118,)
d (1981467, 3) (401590,)
e (960869, 3) (237855,)
f (2510281, 3) (457878,)
g (2236432, 3) (439009,)
h (1970356, 3) (385660,)
i (1142477, 3) (252885,)
j (2926607, 3) (523889,)
k (2228669, 3) (428240,)
l (1609497, 3) (353401,)
m (2102934, 3) (412957,)
n (1606470, 3) (359159,)
o (1152892, 3) (259564,)
p (2193957, 3) (422748,)
q (3108792, 3) (592002,)
r (1449143, 3) (338225,)
s (1267714, 3) (318606,)
t (1705155, 3) (372281,)
u (1362578, 3) (265201,)
v (2755187, 3) (510488,)
w (2466174, 3) (450639,)
x (2937566, 3) (531644,)
y (1764530, 3) (340614,)
z (2929264, 3) (527636,)


In [23]:
# just the letter b
idx = char_matrix[:, 1] == 0
curr_l2_list = l2_list[idx, :]
curr_char_matrix = char_matrix[idx, :]
unique_curr_l2_list = np.unique(curr_l2_list[:, 2])
print('b', curr_l2_list.shape, unique_curr_l2_list.shape)


b (2278449, 3) (437166,)


In [24]:
curr_char_matrix.shape

(2278449, 26)

In [25]:
# how do we dramatically winnow down the search space?
# it feels like I can come up with a way to group each set of letters and then add them. 
# which is what I am already doing. 
# but it's comparison after comparison. 

In [26]:
# Define two arrays
array1 = np.array([1, 2, 3])
array2 = np.array([4, 5])

# Generate combinations using meshgrid and reshape
combinations = np.array(np.meshgrid(array1, array2)).T.reshape(-1, 2)
print(combinations)

[[1 4]
 [1 5]
 [2 4]
 [2 5]
 [3 4]
 [3 5]]


In [27]:
curr_l2_list.shape

(2278449, 3)

In [28]:
unique_curr_l2_list.shape

(437166,)

In [29]:
# split up the list

In [30]:
l2_df.shape

(3213696, 6)

In [31]:
test_l2_df = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [32]:
test_l2_df.shape

(640023, 6)

In [33]:
curr_l2_list = test_l2_df['l2'].to_numpy(dtype = np.int32)

In [34]:
value_splits = list(range(0, curr_l2_list.shape[0] + 10, 10))

In [ ]:
my_sum = 0
for i_v, v in enumerate(value_splits[:-1]):
    my_sum += 1

    combinations = np.array(np.meshgrid(unique_curr_l2_list[:10], unique_curr_l2_list)).T.reshape(-1, 2)
    # combinations = (combinations[:, 0] & combinations[:, 1]) == 0

    print(my_sum)
my_sum

In [ ]:
# this right here is the output
# it's just not faster to split things up because it causes too much enumeration
# the letter selector situation, all over again

In [ ]:
curr_l2_list.shape

In [ ]:
test_l2_df = l2_df.loc[idx, :].reset_index(drop = True)

In [ ]:
test_l2_df.head()

In [ ]:
test_l2_df['l2'].unique().shape

In [ ]:
test_l2_df.shape

In [ ]:
mike = byte_encode_words('mike')
mike

In [ ]:
babb = byte_encode_words('babb')
babb

In [ ]:
bike = byte_encode_words('bike')
bike

In [ ]:
mike & babb

In [ ]:
mike | babb

In [ ]:
mike & bike

In [ ]:
mike | bike

In [ ]:
babb | bike

In [ ]:
working_test_l2_df = test_l2_df.drop_duplicates(subset = 'l2').reset_index()

In [ ]:
output_list = np.zeros(shape = (10000000, 2), dtype = np.int32)
test_l2_all = working_test_l2_df['l2'].to_numpy(dtype = np.int32)
start_pos = 0
for i_l2, l2 in np.ndenumerate(test_l2_all):
    # compare the current l2 to all l2 - this will find all instances
    # indexer for l2, l3, and l4
    positional_idx_l2l3l4 = (test_l2_all & l2) == 0

    # l2 words with different letters
    output_array_w3bw4b = test_l2_all[positional_idx_l2l3l4]    
    
    # l2, l3, l4 accumulated letters
    output_array_l2l3l4 = output_array_w3bw4b | l2
    n_rows = output_array_l2l3l4.shape[0]

    if n_rows > 0:
        temp_output = np.zeros(shape = (n_rows, 2), dtype = np.int32)
        temp_output[:, 0] = l2
        temp_output[:, 1] = output_array_l2l3l4
        output_list[start_pos:start_pos + n_rows, :] = temp_output
        start_pos += n_rows

    if i_l2[0] % 10000 == 0:
        print(i_l2)


    



# BUILD LEVELS 3 THROUGH 5

In [ ]:
l2_df.shape

In [ ]:
l2_all = l2_list[:, 2]

In [ ]:
l2_all.shape

In [ ]:
l2_df_test = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [ ]:
l2_df_test.shape

In [ ]:
l2_all = l2_df_test['l2'].to_numpy(dtype=np.int32)

In [ ]:
# so, now, let's try computing all possible pairs
start_pos = 0
total_output = np.full(shape = (100000000, 5), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df_test.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # compare the current l2 to all l2 - this will find all instances
    # indexer for l2, l3, and l4
    positional_idx_l2l3l4 = (l2_all & l2) == 0

    # combined l2 words and words with different letters
    output_array_w3bw4b = l2_all[positional_idx_l2l3l4]    
    
    # l2, l3, l4 accumulated letters
    output_array_l2l3l4 = output_array_w3bw4b | l2

    # create the temp output
    n_rows_l2l3l4 = output_array_l2l3l4.shape[0]
    if n_rows_l2l3l4 > 0:

        # check against the word_byte_arry for w5b
        
        for w3bw4b, l2l3l4 in zip(output_array_w3bw4b, output_array_l2l3l4):

            positional_idx_l2l3l4l5 = (l2l3l4 & word_byte_array) == 0
            output_array_l2l3l4l5 = word_byte_array[positional_idx_l2l3l4l5]
            n_rows_l2l3l4l5 = output_array_l2l3l4l5.shape[0]

            if n_rows_l2l3l4 > 0:        
                temp_output = np.zeros(shape = (n_rows_l2l3l4l5, 5), dtype = np.int32)
                temp_output[:, 0] = w1b
                temp_output[:, 1] = w2b

                # calcualte w3b and w4b
                # this will look up the 'other' w1b and w2b values
                #w3bw4b = l2_df_test.loc[l2_df_test['l2'] == l2, ['w1b', 'w2b']].to_numpy(dtype = np.int32)                
                temp_output[:, 2] = w3bw4b
        
                # w3b
                #temp_output[:, 2] = w3bw4b[:, 0]
        
                #w4b
                #temp_output[:, 3] = w3bw4b[:, 1]

                #w5b
                temp_output[:, 4] = output_array_l2l3l4l5
        
                total_output[start_pos:start_pos + n_rows_l2l3l4l5, :] = temp_output
        
                start_pos += n_rows_l2l3l4l5        
    
    if i_row % 1000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)

    row_index += 1

In [ ]:
testo = total_output[:start_pos]

In [ ]:
testo.shape

In [ ]:
testo[0, 0] | testo[0, 1] | testo[0, 2] | testo [0, 3]

In [ ]:
np.save('testo.npy', arr = testo)

In [ ]:
testo

In [ ]:
grand_output.shape

In [ ]:
grand_output

In [ ]:
go_df = pd.DataFrame(data = grand_output, columns = ['l2l3l4', 'w5b'])

In [ ]:
testo_unique.shape

In [ ]:
w4_df = pd.DataFrame(data = testo_unique, columns = ['w1b', 'w2b', 'w3b', 'w4b', 'l2', 'l3l4', 'l2l3l4'])

In [ ]:
w4_df.shape

In [ ]:
outcome = pd.merge(left = w4_df, right = go_df)

In [ ]:
outcome.shape

In [ ]:
# add the words
for cn_idx in range(1, 6):
    b_cn = f"w{cn_idx}b"
    w_cn = f"w{cn_idx}"
    outcome[w_cn] = outcome[b_cn].map(word_byte_to_word_dict)

In [ ]:
outcome.head()

In [ ]:
# sort the words
col_names = ['w1', 'w2', 'w3', 'w4', 'w5']
outcome['word_group'] = outcome[col_names].apply(lambda x: ''.join(sorted(x)), axis = 1)

In [ ]:
outcome.head()

In [ ]:
# drop duplicates across the word fields
col_names = ['w1', 'w2', 'w3', 'w4', 'w5', 'word_group']
o_word_df = outcome[col_names].drop_duplicates(subset='word_group')

In [ ]:
o_word_df.shape

In [ ]:
o_word_df.head()

In [ ]:
# so, now, let's try computing all possible pairs
total_output = np.full(shape = (1000000, 9), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3 = (word_byte_array & l2) == 0

    # l3 words with different letters
    output_array_w3b = word_byte_array[positional_idx_l3]    

    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    ## enumerate level 3
    for w3b, l3 in zip(output_array_w3b, output_array_l3):

        # build level 4

        # l4 idx
        positional_idx_l4 = (word_byte_array & l3) == 0

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx_l4]    
        
        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        ## enumerate level 5
        for w4b, l4 in zip(output_array_w4b, output_array_l4):

            # build level 5

            # l5 idx
            positional_idx_l5 = (word_byte_array & l4) == 0
            
            # words with different letters
            output_array_w5b = word_byte_array[positional_idx_l5]    

            if output_array_w5b.size > 0:
                    
                # accumulated letters
                output_array_l5 = output_array_w5b | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b, output_array_l5):

                    temp_list = np.array([w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5], dtype = np.int32)
                    total_output[row_index, :] = temp_list                   
                    row_index += 1


    if i_row % 10000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)
    


# CREATE AND SAVE OUTPUT

In [ ]:
total_output = total_output[:row_index, :]
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5']
l5_df = pd.DataFrame(data = total_output, columns = col_names)


In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.tail()

In [ ]:
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)
